In [1]:
import os 

In [2]:
%pwd

'c:\\Users\\p00za\\Desktop\\Collage projects\\Kidney_Disease_Classification_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\p00za\\Desktop\\Collage projects\\Kidney_Disease_Classification_Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir : Path
    source_url : str
    local_data_file : Path
    unzip_dir : Path

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.Common import read_yaml,create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_url = config.source_url,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

        
        

In [8]:
import os
import urllib.request as request
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.Common import get_size

In [12]:
class DataIngestion:
    def __init__(self, config:DataIngestionConfig):
        self.config = config

    def download_file(self)-> str:
        '''Fetch file from url and save to local directory
        '''

        try:
            dataset_url = self.config.source_url
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading file from :[{dataset_url}] into :[{zip_download_dir}]")

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?/export?format=download&id="
            gdown.download(prefix+file_id, zip_download_dir, quiet=False)

            logger.info(f"File downloaded successfully from {dataset_url} and saved to :[{zip_download_dir}] ") 

        except Exception as e:
            logger.exception(e)
            raise e


    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file to the specified directory
        Function returns None
        """ 
        
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [13]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e    

[2026-05-08 02:15:04,291: INFO: Common: yaml file:config\config.yaml loaded successfully]
[2026-05-08 02:15:04,294: INFO: Common: yaml file:params.yaml loaded successfully]
[2026-05-08 02:15:04,297: INFO: Common: created directory at :artifacts]
[2026-05-08 02:15:04,298: INFO: Common: created directory at :artifacts/data_ingestion]
[2026-05-08 02:15:04,300: INFO: 1287612163: Downloading file from :[https://drive.google.com/file/d/1BOm_Pr25bBfHD96TNNBTE0xA7SfHvzs5/view?usp=sharing] into :[artifacts/data_ingestion/kidney_disease.csv]]


Downloading...
From (original): https://drive.google.com/uc?id=1BOm_Pr25bBfHD96TNNBTE0xA7SfHvzs5
From (redirected): https://drive.google.com/uc?id=1BOm_Pr25bBfHD96TNNBTE0xA7SfHvzs5&confirm=t&uuid=19167fbb-de6e-443b-bc7e-cbe119f47db0
To: c:\Users\p00za\Desktop\Collage projects\Kidney_Disease_Classification_Project\artifacts\data_ingestion\kidney_disease.csv
100%|██████████| 1.63G/1.63G [04:53<00:00, 5.55MB/s]


[2026-05-08 02:20:03,829: INFO: 1287612163: File downloaded successfully from https://drive.google.com/file/d/1BOm_Pr25bBfHD96TNNBTE0xA7SfHvzs5/view?usp=sharing and saved to :[artifacts/data_ingestion/kidney_disease.csv] ]
